In [1]:
import pandas as pd
import numpy as np
import wfdb
import random as rn
import ast
path = '../'
sampling_rate=500
# SCP-ECG文件是一种用于存储心电图数据的文件格式 #new_ptbxl_database.csv存储了标签
Y = pd.read_csv(path+'new_ptbxl_database.csv', index_col='ecg_id',encoding = 'gb2312')
Y

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709,56,1,NaN,63.0,2.0,0.0,CS-12 E,1984/11/9 9:17,sinusrhythmus periphere niederspannung,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243,19,0,NaN,70.0,2.0,0.0,CS-12 E,1984/11/14 12:55,sinusbradykardie sonst normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372,37,1,NaN,69.0,2.0,0.0,CS-12 E,1984/11/15 12:49,sinusrhythmus normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
4,17014,24,0,NaN,82.0,2.0,0.0,CS-12 E,1984/11/15 13:44,sinusrhythmus normales ekg,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
5,17448,19,1,NaN,70.0,2.0,0.0,CS-12 E,1984/11/17 10:43,sinusrhythmus normales ekg,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21830,10520,86,0,NaN,NaN,1.0,2.0,AT-60 3,2001/5/28 7:53,sinusrhythmus lagetyp normal periphere nieders...,...,True,NaN,NaN,NaN,NaN,NaN,NaN,1,records100/21000/21830_lr,records500/21000/21830_hr
21831,11905,55,1,NaN,NaN,1.0,2.0,AT-60 3,2001/5/28 12:49,sinusrhythmus lagetyp normal normales ekg 4.46...,...,True,NaN,NaN,NaN,NaN,NaN,NaN,9,records100/21000/21831_lr,records500/21000/21831_hr
21834,20703,300,0,NaN,NaN,1.0,2.0,AT-60 3,2001/6/5 11:33,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,True,NaN,NaN,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr


In [25]:
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))#取出scp_codes字典中的变量并赋值
Y.scp_codes

ecg_id
1        {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                    {'NORM': 80.0, 'SBRAD': 0.0}
3                      {'NORM': 100.0, 'SR': 0.0}
4                      {'NORM': 100.0, 'SR': 0.0}
5                      {'NORM': 100.0, 'SR': 0.0}
                           ...                   
21830                   {'NORM': 50.0, 'SR': 0.0}
21831                  {'NORM': 100.0, 'SR': 0.0}
21834    {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21836                  {'NORM': 100.0, 'SR': 0.0}
21837                  {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 14801, dtype: object

In [26]:
agg_df = pd.read_csv(path+'scp_statements_257x.csv', index_col=0)#scp_statements_257x.csv'包含了分类任务里面包含哪几类的信息
agg_df = agg_df[agg_df.diagnostic == 1]#选出用于分类中的标签
agg_df

,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_5class,diagnostic_7class,diagnostic_subsubsubclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
NORM,normal ECG,1,NaN,NaN,NORM,NORM,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7
IMI,inferior myocardial infarction,1,NaN,NaN,MI,other,IMI,IMI,Myocardial Infarction,inferior myocardial infarction,161.0,NaN,NaN,NaN
ASMI,anteroseptal myocardial infarction,1,NaN,NaN,MI,ASMI,ASMI,ASMI,Myocardial Infarction,anteroseptal myocardial infarction,165.0,NaN,NaN,NaN
ILMI,inferolateral myocardial infarction,1,NaN,NaN,MI,other,ILMI,ILMI,Myocardial Infarction,inferolateral myocardial infarction,NaN,NaN,NaN,NaN
AMI,anterior myocardial infarction,1,NaN,NaN,MI,AMI,AMI,AMI,Myocardial Infarction,anterior myocardial infarction,160.0,NaN,NaN,NaN
ALMI,anterolateral myocardial infarction,1,NaN,NaN,MI,ALMI,ALMI,ALMI,Myocardial Infarction,anterolateral myocardial infarction,NaN,NaN,NaN,NaN
LMI,lateral myocardial infarction,1,NaN,NaN,MI,other,other,LMI,Myocardial Infarction,lateral myocardial infarction,163.0,NaN,NaN,NaN
IPLMI,inferoposterolateral myocardial infarction,1,NaN,NaN,MI,other,other,IPLMI,Myocardial Infarction,inferoposterolateral myocardial infarction,NaN,NaN,NaN,NaN
IPMI,inferoposterior myocardial infarction,1,NaN,NaN,MI,other,other,IPMI,Myocardial Infarction,inferoposterior myocardial infarction,NaN,NaN,NaN,NaN
PMI,posterior myocardial infarction,1,NaN,NaN,MI,other,other,PMI,Myocardial Infarction,posterior myocardial infarction,162.0,NaN,NaN,NaN


In [27]:
def aggregate_diagnostic(y_dic):#选出当前用于分类的的标签
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_7class)
    return list(set(tmp))

In [28]:
Y['diagnostic_class'] = Y.scp_codes.apply(aggregate_diagnostic)#筛选标签
Y

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,diagnostic_class
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709,56,1,NaN,63.0,2.0,0.0,CS-12 E,1984/11/9 9:17,sinusrhythmus periphere niederspannung,...,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,[NORM]
2,13243,19,0,NaN,70.0,2.0,0.0,CS-12 E,1984/11/14 12:55,sinusbradykardie sonst normales ekg,...,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,[NORM]
3,20372,37,1,NaN,69.0,2.0,0.0,CS-12 E,1984/11/15 12:49,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,[NORM]
4,17014,24,0,NaN,82.0,2.0,0.0,CS-12 E,1984/11/15 13:44,sinusrhythmus normales ekg,...,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,[NORM]
5,17448,19,1,NaN,70.0,2.0,0.0,CS-12 E,1984/11/17 10:43,sinusrhythmus normales ekg,...,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,[NORM]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21830,10520,86,0,NaN,NaN,1.0,2.0,AT-60 3,2001/5/28 7:53,sinusrhythmus lagetyp normal periphere nieders...,...,NaN,NaN,NaN,NaN,NaN,NaN,1,records100/21000/21830_lr,records500/21000/21830_hr,[NORM]
21831,11905,55,1,NaN,NaN,1.0,2.0,AT-60 3,2001/5/28 12:49,sinusrhythmus lagetyp normal normales ekg 4.46...,...,NaN,NaN,NaN,NaN,NaN,NaN,9,records100/21000/21831_lr,records500/21000/21831_hr,[NORM]
21834,20703,300,0,NaN,NaN,1.0,2.0,AT-60 3,2001/6/5 11:33,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,NaN,NaN,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr,[NORM]


In [29]:
def load_ptbxl_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]#wfdb.rdsamp是专门读取ECG信号的函数
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data
def one_hot_7(y_test):
    labels = np.zeros((len(y_test), 7))
    for i in range(len(y_test.values)):
        if len(y_test.values[i])==0:
            continue
        if 'NORM' in y_test.values[i]:
            labels[i,6]=1
        if 'AMI' in y_test.values[i]:
            labels[i,0]=1
        if 'ASMI' in y_test.values[i]:
            labels[i,1]=1
        if 'ALMI' in y_test.values[i]:
            labels[i,2]=1
        if 'IMI' in y_test.values[i]:
            labels[i,3]=1
        if 'ILMI' in y_test.values[i]:
            labels[i,4]=1
        if 'other' in y_test.values[i]:
            labels[i,5]=1
    return labels

In [30]:
strat_fold_train=Y[Y.strat_fold != 10]
strat_fold_train 

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,diagnostic_class
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709,56,1,NaN,63.0,2.0,0.0,CS-12 E,1984/11/9 9:17,sinusrhythmus periphere niederspannung,...,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,[NORM]
2,13243,19,0,NaN,70.0,2.0,0.0,CS-12 E,1984/11/14 12:55,sinusbradykardie sonst normales ekg,...,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,[NORM]
3,20372,37,1,NaN,69.0,2.0,0.0,CS-12 E,1984/11/15 12:49,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,[NORM]
4,17014,24,0,NaN,82.0,2.0,0.0,CS-12 E,1984/11/15 13:44,sinusrhythmus normales ekg,...,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,[NORM]
5,17448,19,1,NaN,70.0,2.0,0.0,CS-12 E,1984/11/17 10:43,sinusrhythmus normales ekg,...,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,[NORM]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21830,10520,86,0,NaN,NaN,1.0,2.0,AT-60 3,2001/5/28 7:53,sinusrhythmus lagetyp normal periphere nieders...,...,NaN,NaN,NaN,NaN,NaN,NaN,1,records100/21000/21830_lr,records500/21000/21830_hr,[NORM]
21831,11905,55,1,NaN,NaN,1.0,2.0,AT-60 3,2001/5/28 12:49,sinusrhythmus lagetyp normal normales ekg 4.46...,...,NaN,NaN,NaN,NaN,NaN,NaN,9,records100/21000/21831_lr,records500/21000/21831_hr,[NORM]
21834,20703,300,0,NaN,NaN,1.0,2.0,AT-60 3,2001/6/5 11:33,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,NaN,NaN,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr,[NORM]


In [36]:
X_train = load_ptbxl_data(strat_fold_train, sampling_rate, path)#心电图信号数据

X_train

MemoryError: Unable to allocate 469. KiB for an array with shape (5000, 12) and data type float64

In [ ]:
X_train.shape

In [ ]:
y_train = strat_fold_train.diagnostic_class
y_train

In [ ]:
import matplotlib.pyplot as plt
plt.plot(X_train)